There are different types of retrievers in LangChain, everything is not required to learn at one go. We will learn here 5 most important types of retrievers.

### Wikipedia Retriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever

C:\Users\koyel\AppData\Local\Temp\ipykernel_7320\2879121110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [4]:
# initialize the retreiever
retriever = WikipediaRetriever(top_k_results=2, lang='en')

- Here during initialization, setting language (parameter `lang`) and required number of top matches (parameter `top_k_results`) are optional.
- If not set, `lang` is by default `en` (english), `top_k_results` is 3.

In [5]:
# define your query
query = "the geopolitical history of India and Pakistan from the perspective of a chinese"

# Get relevant wikipedia documents
docs = retriever.invoke(query) 

- Retriever is a runnable. It has `invoke` property.

In [6]:
print(docs)

[Document(metadata={'title': 'India–Pakistan war of 1971', 'summary': "The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.\nThirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as th

 Not able to understand here properly See below:

In [7]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: \n{doc.page_content}....") # truncate for display


--- Result 1 ---
Content: 
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93

### Vector Store Retriever

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Step 1: Source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models.")
]

In [11]:
# step 2: Initialize embedding Model
embedding_model = OpenAIEmbeddings()

# step 3: Create Chroma Vector Store in memory
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = "my_collection"
)

In [12]:
# step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [13]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [14]:
for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
Chroma is a vector database optimized for LLM-based search.

--Result 2---
LangChain helps developers build LLM applications easily.


In [ ]:
results = vectorstore.similarity_search(query,k=2)

for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
Chroma is a vector database optimized for LLM-based search.

--Result 2---
LangChain helps developers build LLM applications easily.


**If `vectorstore.similarity_search(query, k = 2)` was also giving same result, then why seperate vector store retriever is required?**
- `similarity_search` method can give option for only similarity stratgy, it can't try out different strategies of vector search. This is the major advantage.
- Also above vector store retriever has `invoke` property, so it can be converted to `chains` which is ideal practice for prod setup.

### MMR (Maximum Marginal Relevance)
- In earlier methods, retrevers will return top matches but it may happen that those matches are not much different among themselves to give better coverage and point of views in answer.
- This retriever return those top matches which are having max amount of similarity to query but they themselves are as much dissimilar as possible.

In [16]:
# sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more.")
]

In [17]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [ ]:
# enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type = "mmr",         
    search_kwargs = {"k": 3, "lambda_mult": 1}  
)

- Here parameter `search_type` value `mmr` enables MMR.
- `k` indicates `required number of top results`.
- Parameter `lambda_mult` indicates `relevance-diversity balance`. For `lambda_mult = 0` -> `very diverse result`. This value should be somewhat between 0 and 1. For `lambda_mult = 1`, it will work as normal similarity search, it will not work for fetching relevant yet diverse matches.

In [21]:
query = "What is Langchain?"
results = retriever.invoke(query)

In [20]:
for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
LangChain is used to build LLM based applications.

--Result 2---
LangChain makes it easy to work with LLMs.

--Result 3---
LangChain supports Chroma, FAISS, Pinecone, and more.


We saw results with `lambda_mult = 1`. Next see results with `lambda_mult = 0`.

In [22]:
# enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type = "mmr",         
    search_kwargs = {"k": 3, "lambda_mult": 0}  
)

query = "What is Langchain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
LangChain is used to build LLM based applications.

--Result 2---
Embeddings are vector representations of text.

--Result 3---
MMR helps you get diverse results when doing similarity search.


Now see, results are relevant but diverse from one another.

### Multi-Query Retriever
- Sometimes user asks some ambiguous question which can mean various multiple things. 
- In this type of scenarios, LLM first creates clear queries from the ambiguous query. These clear queries answers user's original query from different point of view.
- Later all answers are merged, and user gets a consolidated answer mentioning different direction of the query.

In [26]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [27]:
# relevant health and wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"})
]

In [28]:
# initialize OpenAI Embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [29]:
# enable MMR in the retriever
similarity_retriever = vectorstore.as_retriever(
    search_type = "similarity",         
    search_kwargs = {"k": 5}   
)

- Observation on this `similarity_retriever` is same as told earlier.

In [30]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(
    search_type = "similarity",       
    search_kwargs = {"k": 5}),
    llm = ChatOpenAI(model="gpt-3.5-turbo") 
)

- Here for this multi query retriever `search_type = "similarity"` enables MMR.
- We are adding parameter `llm` to split ambiguous query.

In [31]:
# retriever results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [32]:
for i, doc in enumerate(similarity_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
LangChain is used to build LLM based applications.

--Result 2---
LangChain makes it easy to work with LLMs.

--Result 3---
LangChain supports Chroma, FAISS, Pinecone, and more.

--Result 4---
Chroma is used to store and search document embeddings.

--Result 5---
Embeddings are vector representations of text.


In [33]:
for i, doc in enumerate(multiquery_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
LangChain is used to build LLM based applications.

--Result 2---
LangChain makes it easy to work with LLMs.

--Result 3---
LangChain supports Chroma, FAISS, Pinecone, and more.

--Result 4---
Chroma is used to store and search document embeddings.

--Result 5---
Embeddings are vector representations of text.


### Contextual Compression Retriever

In [36]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [34]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [35]:
# initialize OpenAI Embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [38]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [37]:
# setup the Compressor using an LLM
llm = ChatOpenAI(model='gpt-3.5-turbo')
compressor = LLMChainExtractor.from_llm(llm)

In [39]:
# create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [40]:
# query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [41]:
for i, doc in enumerate(compressed_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)


--Result 1---
Photosynthesis is the process by which green plants convert sunlight into energy.

--Result 2---
The chlorophyll in plant cells captures sunlight during photosynthesis.


- TODO:
    - Add concept of OCR.
    - Add how reranking works on top k searches.
    - Learn other strategies of vector search apart from similarity search.
    - Revisit how chroma and faiss dbs are storing here. If not, then from where retrieval is happening.
    - In multi query retriever, revisit how and wgere multiple queries are getting generated.
    - When contextual compression retriever is used? Add more documentation overall.
    - Paste here documentation link to refer other retrievers and for our easy access as well.